In [6]:
# create demand file
import pandas as pd
import pytz

# Read CSV file with pandas
df = pd.read_csv('../data/raw/demand_2025.csv')

# Remove rows where 'Actual Total Load (MW)' is '-'
df = df[df['Actual Total Load (MW)'] != '-']

# Convert 'Actual Total Load (MW)' to float
df['Actual Total Load (MW)'] = df['Actual Total Load (MW)'].astype(float)

# Prepare timezone
cet = pytz.timezone('Europe/Amsterdam')

# Parse and convert time to UTC
def parse_to_utc(time_str):
    # Get time part after dash and remove timezone info
    time_part = time_str.split('-')[0].strip()
    time_part = time_part.replace(' (CET)', '').replace(' (CEST)', '').strip()
    dt_local = pd.to_datetime(time_part, format='%d/%m/%Y %H:%M')
    dt_local = cet.localize(dt_local, is_dst=None)
    dt_utc = dt_local.astimezone(pytz.UTC)
    return dt_utc.strftime('%Y-%m-%d %H:%M:%S')

df['time_utc'] = df['MTU (CET/CEST)'].apply(parse_to_utc)

# Select and save relevant columns
df_out = df[['time_utc', 'Actual Total Load (MW)']].rename(columns={'Actual Total Load (MW)': 'demand_mw'})
df_out.set_index('time_utc', inplace=True)
df_out.to_csv('../data/clean/demand.csv', index=True)

In [1]:
import pandas as pd

# Define file info for each production type
prod_info = [
    {
        "filename": "../data/raw/zon-2025-uur-data.csv",
        "colname": "solar_production_mw"
    },
    {
        "filename": "../data/raw/zeewind-2025-uur-data.csv",
        "colname": "wind_offshore_production_mw"
    },
    {
        "filename": "../data/raw/wind-2025-uur-data.csv",
        "colname": "wind_onshore_production_mw"
    }
]

dfs = []
for info in prod_info:
    df = pd.read_csv(info["filename"]).filter(["validfrom (UTC)", "volume (kWh)"])
    df.set_index("validfrom (UTC)", inplace=True)
    df.index = pd.to_datetime(df.index)
    df.columns = [info["colname"]]
    df = df.resample('15min').ffill() / 1000
    df.index.name = 'time_utc'
    dfs.append(df)

# Join all dataframes on time_utc
renewable_production = dfs[0].join(dfs[1:], how='left')

renewable_production.to_csv('../data/clean/renewable_production_2025.csv', index=True)

In [2]:
import pandas as pd

renewable_production = pd.read_csv('../data/clean/renewable_production_2025.csv', index_col='time_utc', parse_dates=True)
demand = pd.read_csv('../data/clean/demand.csv', index_col='time_utc', parse_dates=True)

# join
timeseries_data = renewable_production.join(demand, on='time_utc', how='left')

timeseries_data.to_csv('../data/clean/timeseries_data_2025.csv', index=True)

In [ ]:
# create demand file
import pandas as pd
import pytz

# Read CSV file with pandas
df = pd.read_csv('../data/raw/energy_prices_2025.csv')

# Convert 'Day-ahead Price (EUR/MWh)' to float
df["Day-ahead Price (EUR/MWh)"] = df['Day-ahead Price (EUR/MWh)'].astype(float)

# Prepare timezone
cet = pytz.timezone('Europe/Amsterdam')

# Parse and convert time to UTC
def parse_to_utc(time_str):
    # Get time part after dash and remove timezone info
    time_part = time_str.split('-')[0].strip()
    time_part = time_part.replace(' (CET)', '').replace(' (CEST)', '').strip()
    dt_local = pd.to_datetime(time_part, format='%d/%m/%Y %H:%M:%S')
    dt_local = cet.localize(dt_local, is_dst=None)
    dt_utc = dt_local.astimezone(pytz.UTC)
    return dt_utc.strftime('%Y-%m-%d %H:%M:%S')

df['time_utc'] = df['MTU (CET/CEST)'].apply(parse_to_utc)

# Select and save relevant columns
df_out = df[['time_utc', 'Day-ahead Price (EUR/MWh)']].rename(columns={'Day-ahead Price (EUR/MWh)': 'price_eur_mwh'})
df_out.set_index('time_utc', inplace=True)
df_out.to_csv('../data/clean/price.csv', index=True)